# Kaggle llama.cpp backend runner

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/kaggle-llamacpp-backend.git"
REPO_DIR = Path("/kaggle/working/kaggle-llamacpp-backend")

if not REPO_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL:
        raise ValueError("Ubah REPO_URL dulu ke repo GitHub kamu.")
    subprocess.run(["git","clone","--depth","1",REPO_URL,str(REPO_DIR)], check=True)

subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(REPO_DIR/"requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR/"src"))

from kaggle_llamacpp import ensure_llamacpp_cuda
ensure_llamacpp_cuda()


In [ ]:
from kaggle_llamacpp import download_assets

MODEL_URL = "https://huggingface.co/mradermacher/gemma-4-26B-A4B-Heretic-Stable-i1-GGUF/resolve/main/gemma-4-26B-A4B-Heretic-Stable.i1-Q4_K_M.gguf"
MMPROJ_URL = ""  # optional vision projector. Kosong = text-only.

download_assets(
    model_url=MODEL_URL,
    mmproj_url=MMPROJ_URL,
    backend="aria2",
)


In [ ]:
from kaggle_llamacpp import (
    ServerConfig, start_llama_server, wait_until_ready,
    test_models_endpoint, test_chat_completion, test_vision_completion, print_status
)

cfg = ServerConfig(
    model_alias="local",
    port=8080,
    ctx_size=8192,      # 2048 aman, 8192 stabil, 20000 eksperimen
    batch_size=256,
    ubatch_size=64,
    gpu_layers=999,
    split_mode="layer",
    tensor_split="1,1",
    cuda_visible_devices="0,1",
    threads=4,
    parallel=1,
    reasoning_format="none",
    enable_thinking=False,
    enable_vision="auto",   # otomatis pakai mmproj kalau Cell 2 mengisi MMPROJ_URL
    mmproj_offload=True,
    no_warmup=False,
    api_key=None,
)

start_llama_server(cfg)

if not wait_until_ready(port=cfg.port, timeout_s=900):
    print_status(lines=240)
    raise RuntimeError("Server belum ready.")

test_models_endpoint(port=cfg.port)

print("\nMODEL ID UNTUK APLIKASI:", cfg.model_alias)
print("LOCAL BASE URL:", f"http://127.0.0.1:{cfg.port}/v1")

text = test_chat_completion(
    "Say hello in one short sentence. Do not explain your reasoning.",
    port=cfg.port,
    model=cfg.model_alias,
    max_tokens=64,
)
print("\nFINAL CHAT:", text)

vision = test_vision_completion(
    port=cfg.port,
    model=cfg.model_alias,
    skip_if_no_mmproj=True,
)
if vision:
    print("\nFINAL VISION:", vision)


In [ ]:
from kaggle_llamacpp import start_ngrok_tunnel

public_url = start_ngrok_tunnel(
    port=8080,
    secret_name="ngrok",  # ubah kalau nama secret kamu beda
)

print("\nOpenAI-compatible Base URL:")
print(public_url + "/v1")
print("\nModel ID:")
print("local")
